# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [ ]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(18):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-270m-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

print(transcoder_paths)  # verify all 18 are present

{0: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_0_width_16k_l0_small/params.safetensors', 1: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_1_width_16k_l0_small/params.safetensors', 2: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_2_width_16k_l0_small/params.safetensors', 3: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_all/layer_3_width_16k_l0_small/params.safetensors', 4: '/teamspace/studios/this_studio/.cache/huggingface/hub/models--google--gemma-scope-2-270m-pt/snapshots/b218cd5d69dc2fa71cff448b68d625e6c9702d49/transcoder_a

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-270m-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-270m",
    transcoders=transcoder_set,       # ← our manually loaded TranscoderSet
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

Loaded pretrained model google/gemma-3-270m into HookedTransformer


# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-270m"  # ← set this
DESCRIBED_LAYERS = {5, 9, 12, 15}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [9]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [10]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

TOP 10 OVERALL (all layers)
 1. L17 F 1049  influence= 11.3329    undescribed
 2. L16 F14713  influence= 11.2031    undescribed
 3. L16 F 9362  influence= 10.9282    undescribed
 4. L16 F13993  influence= 10.9144    undescribed
 5. L17 F12205  influence= 10.8671    undescribed
 6. L17 F 4359  influence= 10.8653    undescribed
 7. L16 F13891  influence= 10.7674    undescribed
 8. L16 F 5212  influence= 10.7454    undescribed
 9. L17 F 2753  influence= 10.5294    undescribed
10. L17 F 3844  influence= 10.4468    undescribed


### top ten from only described layers

In [11]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (5, 9, 12, 15)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (5, 9, 12, 15)
 1. L15 F 2094  influence=  6.7952
 2. L15 F14225  influence=  6.6352
 3. L15 F 7651  influence=  6.3633
 4. L15 F13723  influence=  6.0249
 5. L15 F 3129  influence=  5.6426
 6. L15 F15858  influence=  5.1249
 7. L15 F12182  influence=  5.0907
 8. L15 F 7490  influence=  4.9146
 9. L15 F14104  influence=  4.5888
10. L15 F 6769  influence=  4.5676


### top ten from every other layer

In [12]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 5, 9, 12, 15)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

TOP 10 FROM OTHER LAYERS (not 5, 9, 12, 15)
 1. L17 F 1049  influence= 11.3329
 2. L16 F14713  influence= 11.2031
 3. L16 F 9362  influence= 10.9282
 4. L16 F13993  influence= 10.9144
 5. L17 F12205  influence= 10.8671
 6. L17 F 4359  influence= 10.8653
 7. L16 F13891  influence= 10.7674
 8. L16 F 5212  influence= 10.7454
 9. L17 F 2753  influence= 10.5294
10. L17 F 3844  influence= 10.4468


### summary

In [13]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

Total unique transcoder features: 3052
Features in described layers:     371
Features in other layers:         2681


# intervention

### all 10 features suppressed

In [14]:
# Find "it" in the second line
baseline = "A rhyming couplet:\nHe saw a carrot and had to grab it,\nHe saw a carrot and had to grab it,"
tokens = model.to_tokens(baseline)
token_strings = [model.tokenizer.decode([t]) for t in tokens[0]]

print("Tokens:")
for i, ts in enumerate(token_strings):
    print(f"  {i}: {repr(ts)}")

# Find the LAST "it" (the one at the end of the second line)
it_positions = [i for i, ts in enumerate(token_strings) if ts.strip() == 'it']
print(f"\n'it' appears at positions: {it_positions}")
rhyme_pos = it_positions[-1]  # last one
print(f"Rhyme token 'it' at position {rhyme_pos}")

Tokens:
  0: '<bos>'
  1: 'A'
  2: ' rh'
  3: 'yming'
  4: ' couple'
  5: 't'
  6: ':'
  7: '\n'
  8: 'He'
  9: ' saw'
  10: ' a'
  11: ' carrot'
  12: ' and'
  13: ' had'
  14: ' to'
  15: ' grab'
  16: ' it'
  17: ','
  18: '\n'
  19: 'He'
  20: ' saw'
  21: ' a'
  22: ' carrot'
  23: ' and'
  24: ' had'
  25: ' to'
  26: ' grab'
  27: ' it'
  28: ','

'it' appears at positions: [16, 27]
Rhyme token 'it' at position 27


In [16]:
import json

# Load the rhyme step
with open("./graphs/gemma-3-270m/step-08-it.json") as f:
    data = json.load(f)

# Extract top features by influence
features_by_influence = {}
for node in data.get("nodes", []):
    if "transcoder" not in node.get("feature_type", ""):
        continue
    layer = node.get("layer")
    feat = node.get("feature")
    inf = abs(node.get("influence", 0) or 0)
    
    features_by_influence[(layer, feat)] = inf

# Top 10
top_10 = sorted(features_by_influence.items(), key=lambda x: -x[1])[:10]

print("TOP 10 FEATURES DRIVING THE RHYME (step-08-it):\n")
for (layer, feat), score in top_10:
    print(f"L{layer} F{feat}: {score:.4f}")

TOP 10 FEATURES DRIVING THE RHYME (step-08-it):

L16 F118710919: 0.8002
L16 F29203886: 0.7997
L17 F66326385: 0.7993
L16 F42684163: 0.7988
L16 F77470111: 0.7984
L16 F11207728: 0.7980
L17 F97391928: 0.7975
L16 F26350153: 0.7971
L16 F18480143: 0.7962
L16 F84194759: 0.7958


In [17]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [18]:
# 1. Cast everything to int during tuple creation to avoid downstream errors
features_to_intervene = [(int(layer), int(feat)) for (layer, feat), _ in top_10]

# 2. Map global IDs to local indices (using the 16k width logic we discussed)
# Note: Ensure TARGET_POS is also an integer
intervention_tuples = [(layer, int(TARGET_POS), feat % 16384, 0.0) for layer, feat in features_to_intervene]

print("\n" + "=" * 70)
print("INTERVENTION TUPLES (top 10 features, suppressed at target position)")
print("=" * 70)

for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    # Now :d will work because they are guaranteed integers
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val:.1f}")


INTERVENTION TUPLES (top 10 features, suppressed at target position)
 1. Layer 16, Position  -1, Feature  8839, Value 0.0
 2. Layer 16, Position  -1, Feature  7598, Value 0.0
 3. Layer 17, Position  -1, Feature  3953, Value 0.0
 4. Layer 16, Position  -1, Feature  3843, Value 0.0
 5. Layer 16, Position  -1, Feature  6559, Value 0.0
 6. Layer 16, Position  -1, Feature  1072, Value 0.0
 7. Layer 17, Position  -1, Feature  5432, Value 0.0
 8. Layer 16, Position  -1, Feature  4681, Value 0.0
 9. Layer 16, Position  -1, Feature 15375, Value 0.0
10. Layer 16, Position  -1, Feature 13767, Value 0.0


In [19]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 10 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")


GENERATING WITH ORIGINAL FEATURES (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,

GENERATING WITH ALL 10 FEATURES SUPPRESSED


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
っております
He saw a carrot and had to grab


In [25]:
import json
import glob

prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

print("BASELINE (no intervention):")
model.reset_hooks()
baseline = model.generate(prompt, max_new_tokens=15, do_sample=False)
print(baseline)
print()

# Test step 0, 4, 8 (spread across sequence)
test_steps = [0, 4, 8]

for step_num in test_steps:
    graph_files = glob.glob(f"./graphs/gemma-3-270m/step-{step_num:02d}-*.json")
    
    if not graph_files:
        print(f"Step {step_num}: No file found\n")
        continue
    
    print(f"Step {step_num}:")
    # Just try suppressing the top described features instead
    # These are easier to work with
    
    top_features = [
        (16, 8839),   # from described layers
        (16, 7598),
        (17, 3953),
    ]
    
    intervention = [(layer, 0, feat, 0.0) for layer, feat in top_features]
    
    model.reset_hooks()
    suppressed = model.feature_intervention_generate(prompt, intervention, do_sample=False, verbose=False)[0]
    
    # rhymes = "grab it" in suppressed or "crap it" in suppressed
    # print(f"  Rhymes: {rhymes}")
    print(f"  Output: {suppressed[:200]}\n")

BASELINE (no intervention):


  0%|          | 0/15 [00:00<?, ?it/s]

A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,
He saw a carrot

Step 0:
  Output: A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,

Step 4:
  Output: A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,

Step 8:
  Output: A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,



In [29]:
# ══════════════════════════════════════════════════════════════════════════════
# CIRCUIT TRACING: Planning vs Execution Stage Analysis
# Mirrors the poetry experiment from the Anthropic attribution-graphs paper.
# Paste this as new cells after your existing "intervention" section.
# ══════════════════════════════════════════════════════════════════════════════

import json
import glob
import re
import requests
from collections import defaultdict
from pathlib import Path


# ── CONFIG ────────────────────────────────────────────────────────────────────
GRAPH_DIR = "./graphs/gemma-3-270m"          # same as your existing cells
RHYME_TOKEN = "it"                            # the token you're rhyming on
RHYME_STEP = 8                                # step index that produces "it"
TOP_N = 15                                    # features to track per step
MODEL_ID = "gemma-scope-2-270m-pt"           # for Neuronpedia lookups
FEAT_WIDTH = "16k"


# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Per-step feature extraction
# Builds a timeline: {step -> [(layer, feat, influence, token_str), ...]}
# ══════════════════════════════════════════════════════════════════════════════

def parse_node_ids(node):
    """Return (layer, local_feat) from node dict, handling both id formats."""
    # Try jsNodeId first (format: "LAYER_FEAT-TOKEN_POS")
    js = node.get("jsNodeId", "")
    m = re.match(r"^(\d+)_(\d+)-", js)
    if m:
        return int(m.group(1)), int(m.group(2))
    # Fallback: use node["layer"] and node["feature"] directly
    layer = node.get("layer")
    feat  = node.get("feature")
    if layer is not None and feat is not None:
        return int(layer), int(feat)
    return None, None


step_timelines = {}   # step_idx -> list of dicts

for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    fname = Path(fpath).stem                  # e.g. "step-08-it"
    m = re.match(r"step-(\d+)-(.+)", fname)
    if not m:
        continue
    step_idx  = int(m.group(1))
    token_str = m.group(2).replace("_", " ")

    with open(fpath) as f:
        data = json.load(f)

    rows = []
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        inf = node.get("influence") or 0
        if inf == 0:
            continue
        layer, feat = parse_node_ids(node)
        if layer is None:
            continue
        rows.append({
            "step":      step_idx,
            "token":     token_str,
            "layer":     layer,
            "feat":      feat,
            "influence": abs(inf),
            "raw_inf":   inf,
        })

    rows.sort(key=lambda x: -x["influence"])
    step_timelines[step_idx] = rows[:TOP_N]

print(f"Loaded {len(step_timelines)} steps: {sorted(step_timelines)}")
for s, rows in sorted(step_timelines.items()):
    tok = rows[0]["token"] if rows else "?"
    print(f"  step {s:02d} '{tok}': {len(rows)} top features")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Feature timeline: track each feature across all steps
# Produces feature_timeline[feat_key] = {step: influence}
# ══════════════════════════════════════════════════════════════════════════════

feature_timeline = defaultdict(lambda: defaultdict(float))
all_steps_tokens = {}   # step -> token_str

for step_idx, rows in step_timelines.items():
    if rows:
        all_steps_tokens[step_idx] = rows[0]["token"]
    for r in rows:
        key = (r["layer"], r["feat"])
        feature_timeline[key][step_idx] = r["influence"]

# Identify features that appear in the rhyme step
rhyme_step_features = set()
if RHYME_STEP in step_timelines:
    for r in step_timelines[RHYME_STEP]:
        rhyme_step_features.add((r["layer"], r["feat"]))

print(f"\nUnique features across all steps: {len(feature_timeline)}")
print(f"Features active at rhyme step {RHYME_STEP}: {len(rhyme_step_features)}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Early-spike detection (planning features)
# A feature is a "planning" feature if its peak influence is BEFORE the
# rhyme step (mirrors the paper's "rhyme-feature activates early" finding).
# ══════════════════════════════════════════════════════════════════════════════

all_step_indices = sorted(step_timelines.keys())

planning_features  = []   # peak before rhyme step
execution_features = []   # peak at or after rhyme step

for feat_key, step_inf in feature_timeline.items():
    if not step_inf:
        continue
    peak_step = max(step_inf, key=step_inf.get)
    peak_val  = step_inf[peak_step]
    rhyme_val = step_inf.get(RHYME_STEP, 0.0)
    span      = sorted(step_inf.keys())

    entry = {
        "feat_key":   feat_key,
        "peak_step":  peak_step,
        "peak_val":   peak_val,
        "rhyme_val":  rhyme_val,
        "first_step": span[0],
        "last_step":  span[-1],
        "n_steps":    len(span),
    }

    if peak_step < RHYME_STEP:
        planning_features.append(entry)
    else:
        execution_features.append(entry)

planning_features.sort(key=lambda x: -x["peak_val"])
execution_features.sort(key=lambda x: -x["peak_val"])

print("=" * 70)
print(f"PLANNING FEATURES  (peak before step {RHYME_STEP}): {len(planning_features)}")
print("=" * 70)
for e in planning_features[:10]:
    l, f = e["feat_key"]
    print(f"  L{l:2d} F{f:5d}  peak_step={e['peak_step']}  peak_inf={e['peak_val']:.4f}  "
          f"rhyme_inf={e['rhyme_val']:.4f}  active_steps={e['n_steps']}")

print()
print("=" * 70)
print(f"EXECUTION FEATURES (peak at step {RHYME_STEP} or after): {len(execution_features)}")
print("=" * 70)
for e in execution_features[:10]:
    l, f = e["feat_key"]
    print(f"  L{l:2d} F{f:5d}  peak_step={e['peak_step']}  peak_inf={e['peak_val']:.4f}  "
          f"rhyme_inf={e['rhyme_val']:.4f}  active_steps={e['n_steps']}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Rhyme-circuit candidate features
# Features that: (a) appear in the rhyme step AND (b) were already active
# at least 2 steps earlier → strongest planning signal
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("RHYME-CIRCUIT CANDIDATES  (active at rhyme AND spiked earlier)")
print("=" * 70)

candidates = []
for e in planning_features:
    if e["feat_key"] in rhyme_step_features:
        candidates.append(e)

candidates.sort(key=lambda x: -(x["peak_val"] + x["rhyme_val"]))

for e in candidates[:15]:
    l, f = e["feat_key"]
    steps_before = RHYME_STEP - e["first_step"]
    print(f"  L{l:2d} F{f:5d}  first_active=step{e['first_step']}  "
          f"({steps_before} steps before rhyme)  "
          f"peak={e['peak_val']:.4f} @ step{e['peak_step']}  "
          f"rhyme_inf={e['rhyme_val']:.4f}")

print(f"\nTotal rhyme-circuit candidates: {len(candidates)}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Per-step top features printed as a table
# Lets you see which layer/features dominate at each generation step
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("PER-STEP TOP FEATURES (step × layer × feature × influence)")
print("=" * 70)

for step_idx in sorted(step_timelines.keys()):
    rows = step_timelines[step_idx]
    tok  = rows[0]["token"] if rows else "?"
    marker = "  ← RHYME" if step_idx == RHYME_STEP else ""
    print(f"\nstep {step_idx:02d} '{tok}'{marker}")
    for r in rows[:5]:
        print(f"    L{r['layer']:2d} F{r['feat']:5d}  inf={r['influence']:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Neuronpedia feature descriptions
# Fetches labels for top candidates; works for described layers only.
# Set FETCH_DESCRIPTIONS = False to skip if offline / no API access.
# ══════════════════════════════════════════════════════════════════════════════

FETCH_DESCRIPTIONS = False
NP_BASE = "https://www.neuronpedia.org/api/feature"

def fetch_description(layer, feat, model_id=MODEL_ID, width=FEAT_WIDTH):
    """Return Neuronpedia description string or None."""
    # Neuronpedia SAE ID for gemma-scope transcoders
    sae_id = f"{layer}-gemmascope-2-transcoder-{width}"
    url = f"{NP_BASE}/{model_id}/{sae_id}/{feat}"
    try:
        r = requests.get(url, timeout=8)
        if r.status_code != 200:
            return None
        d = r.json()
        # description lives under explanations[0].description
        exps = d.get("explanations") or []
        if exps:
            return exps[0].get("description")
        return d.get("label") or d.get("description")
    except Exception:
        return None


if FETCH_DESCRIPTIONS:
    print("=" * 70)
    print("NEURONPEDIA DESCRIPTIONS FOR TOP RHYME-CIRCUIT CANDIDATES")
    print("=" * 70)

    # Query top candidates + top execution features at rhyme step
    query_set = [(e["feat_key"], "planning")  for e in candidates[:8]] + \
                [(e["feat_key"], "execution") for e in execution_features[:5]
                 if e["feat_key"] in rhyme_step_features]

    for (layer, feat), stage in query_set:
        desc = fetch_description(layer, feat)
        label = desc if desc else "(no description)"
        print(f"  [{stage:9s}] L{layer:2d} F{feat:5d}  →  {label}")
else:
    print("(Neuronpedia fetch disabled — set FETCH_DESCRIPTIONS = True to enable)")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Temporal clustering summary
# Groups features into EARLY (planning), MID, LATE (execution) bands
# and shows which layers dominate each band.
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter

n_steps = len(all_step_indices)
early_cutoff = RHYME_STEP // 3            # first third
mid_cutoff   = (2 * RHYME_STEP) // 3     # second third

bands = {"EARLY (planning)": [], "MID (buildup)": [], "LATE (execution)": []}

for feat_key, step_inf in feature_timeline.items():
    if not step_inf:
        continue
    peak = max(step_inf, key=step_inf.get)
    if peak <= early_cutoff:
        bands["EARLY (planning)"].append(feat_key)
    elif peak <= mid_cutoff:
        bands["MID (buildup)"].append(feat_key)
    else:
        bands["LATE (execution)"].append(feat_key)

print("=" * 70)
print("TEMPORAL CLUSTERING SUMMARY")
print(f"  early ≤ step {early_cutoff}  |  mid ≤ step {mid_cutoff}  |  late > step {mid_cutoff}")
print("=" * 70)

for band_name, feats in bands.items():
    layer_counts = Counter(layer for layer, _ in feats)
    top_layers = layer_counts.most_common(5)
    print(f"\n{band_name}:  {len(feats)} features")
    print(f"  Top layers: " + "  ".join(f"L{l}×{c}" for l, c in top_layers))
    # show the highest-influence feature per band
    top_in_band = sorted(
        feats,
        key=lambda lf: max(feature_timeline[lf].values()),
        reverse=True
    )[:3]
    for lf in top_in_band:
        peak_inf = max(feature_timeline[lf].values())
        peak_s   = max(feature_timeline[lf], key=feature_timeline[lf].get)
        print(f"    L{lf[0]:2d} F{lf[1]:5d}  peak_inf={peak_inf:.4f} @ step{peak_s}")

Loaded 19 steps: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  step 00 'He': 15 top features
  step 01 'saw': 15 top features
  step 02 'a': 15 top features
  step 03 'carrot': 15 top features
  step 04 'and': 15 top features
  step 05 'had': 15 top features
  step 06 'to': 15 top features
  step 07 'grab': 15 top features
  step 08 'it': 15 top features
  step 09 ',': 15 top features
  step 11 'He': 15 top features
  step 12 'saw': 15 top features
  step 13 'a': 15 top features
  step 14 'carrot': 15 top features
  step 15 'and': 15 top features
  step 16 'had': 15 top features
  step 17 'to': 15 top features
  step 18 'grab': 15 top features
  step 19 'it': 15 top features

Unique features across all steps: 263
Features active at rhyme step 8: 15
PLANNING FEATURES  (peak before step 8): 110
  L16 F 4457  peak_step=1  peak_inf=0.8005  rhyme_inf=0.0000  active_steps=1
  L17 F15738  peak_step=3  peak_inf=0.8004  rhyme_inf=0.0000  active_steps=1
  L16 F11845  peak_

In [32]:
# Check if planning features exist BEFORE the rhyme step
rhyme_step_features = {}
if 8 in step_timelines:
    for r in step_timelines[8]:
        key = (r["layer"], r["feat"])
        rhyme_step_features[key] = r["influence"]

print("=" * 70)
print("DID THESE FEATURES SPIKE EARLY? (planning signal)")
print("=" * 70)

early_spikes = []
for feat_key, rhyme_inf in rhyme_step_features.items():
    if feat_key not in feature_timeline:
        continue
    step_inf = feature_timeline[feat_key]
    
    # Was this feature active BEFORE step 8?
    pre_rhyme_steps = {s: v for s, v in step_inf.items() if s < 8}
    if pre_rhyme_steps:
        pre_peak = max(pre_rhyme_steps.values())
        pre_step = max(pre_rhyme_steps, key=pre_rhyme_steps.get)
        early_spikes.append({
            "feat": feat_key,
            "pre_peak": pre_peak,
            "pre_step": pre_step,
            "rhyme_inf": rhyme_inf,
            "ratio": pre_peak / rhyme_inf if rhyme_inf > 0 else float('inf'),
        })

early_spikes.sort(key=lambda x: -x["pre_peak"])

for e in early_spikes[:10]:
    l, f = e["feat"]
    print(f"L{l:2d} F{f:5d}  peaked @ step{e['pre_step']} (inf={e['pre_peak']:.4f})  "
          f"→  rhyme_step_inf={e['rhyme_inf']:.4f}  ratio={e['ratio']:.2f}x")

print(f"\nTotal features that spiked before rhyme: {len(early_spikes)} / {len(rhyme_step_features)}")

DID THESE FEATURES SPIKE EARLY? (planning signal)
L17 F11499  peaked @ step7 (inf=0.7968)  →  rhyme_step_inf=0.7993  ratio=1.00x

Total features that spiked before rhyme: 1 / 15


In [42]:
import json
from pathlib import Path

GRAPH_DIR = "./graphs/gemma-3-270m"

# Load the step-19 graph (where the rhyme is produced)
with open(f"{GRAPH_DIR}/step-19-it.json") as f:
    rhyme_graph = json.load(f)

print("=" * 70)
print("STEP 19 ATTRIBUTION GRAPH (rhyme token 'it')")
print("=" * 70)

# Look at the target node (the output logit for 'it')
target_node = None
for node in rhyme_graph.get("nodes", []):
    if node.get("is_target_logit"):
        target_node = node
        print(f"Target node: {node.get('node_id')}")
        print(f"  Influence: {node.get('influence')}")
        break

# Find edges pointing TO the target
print("\n" + "=" * 70)
print("EDGES FEEDING INTO THE RHYME (step 19)")
print("=" * 70)

incoming_edges = [e for e in rhyme_graph.get("edges", []) 
                  if e.get("target") == target_node.get("node_id")]

print(f"\nTotal edges feeding into rhyme target: {len(incoming_edges)}")
print("\nTop contributing edges (by weight):")

incoming_edges_sorted = sorted(incoming_edges, key=lambda e: -abs(e.get("weight", 0)))

for edge in incoming_edges_sorted[:15]:
    source = edge.get("source")
    target = edge.get("target")
    weight = edge.get("weight", 0)
    print(f"  {source} → {target}: weight={weight:.4f}")

# Now find: which STEPS do these source features come from?
print("\n" + "=" * 70)
print("WHERE DO THESE FEATURES COME FROM?")
print("=" * 70)

source_features = set()
for edge in incoming_edges:
    source_id = edge.get("source")
    source_features.add(source_id)

print(f"\nUnique source features feeding rhyme: {len(source_features)}")
print("\nSource node IDs:")
for src in list(source_features)[:10]:
    print(f"  {src}")

# Parse the node IDs to see which steps/layers they're from
# Node format might be: "layer_feature-position" or similar
print("\n" + "=" * 70)
print("TRACING BACK: Which steps feed the rhyme?")
print("=" * 70)

# Find these source nodes in the graph
step_origins = {}
for node in rhyme_graph.get("nodes", []):
    node_id = node.get("node_id")
    if node_id in source_features:
        # Try to parse which step this came from
        # This depends on the graph format
        print(f"Source node: {node_id}")
        print(f"  Layer: {node.get('layer')}")
        print(f"  Feature: {node.get('feature')}")
        print(f"  Influence: {node.get('influence'):.4f}")

STEP 19 ATTRIBUTION GRAPH (rhyme token 'it')
Target node: 19_625_37
  Influence: None

EDGES FEEDING INTO THE RHYME (step 19)

Total edges feeding into rhyme target: 0

Top contributing edges (by weight):

WHERE DO THESE FEATURES COME FROM?

Unique source features feeding rhyme: 0

Source node IDs:

TRACING BACK: Which steps feed the rhyme?


In [44]:
import json
from pathlib import Path

GRAPH_DIR = "./graphs/gemma-3-270m"

# Load and inspect the actual structure
with open(f"{GRAPH_DIR}/step-19-it.json") as f:
    data = json.load(f)

print("=" * 70)
print("GRAPH STRUCTURE INSPECTION")
print("=" * 70)

print(f"\nTop-level keys: {list(data.keys())}")
print(f"\nMetadata: {data.get('metadata', {})}")

print(f"\n# Nodes: {len(data.get('nodes', []))}")
print(f"# Edges: {len(data.get('edges', []))}")

# Look at a few nodes
print("\n" + "=" * 70)
print("SAMPLE NODES")
print("=" * 70)

nodes = data.get("nodes", [])
for i, node in enumerate(nodes[:5]):
    print(f"\nNode {i}:")
    for key, val in node.items():
        if key != "metadata":
            print(f"  {key}: {val}")

# Look at a few edges
print("\n" + "=" * 70)
print("SAMPLE EDGES")
print("=" * 70)

edges = data.get("edges", [])
for i, edge in enumerate(edges[:5]):
    print(f"\nEdge {i}:")
    for key, val in edge.items():
        print(f"  {key}: {val}")

# Find target logit node
print("\n" + "=" * 70)
print("TARGET LOGIT NODE")
print("=" * 70)

target_nodes = [n for n in nodes if n.get("is_target_logit")]
print(f"Target logit nodes: {len(target_nodes)}")
if target_nodes:
    for node in target_nodes:
        print(f"\n{node.get('node_id')}:")
        print(f"  influence: {node.get('influence')}")
        print(f"  is_target_logit: {node.get('is_target_logit')}")

# Look at high-influence nodes (the actual drivers)
print("\n" + "=" * 70)
print("TOP INFLUENCE NODES (excluding target)")
print("=" * 70)

non_target = [n for n in nodes if not n.get("is_target_logit")]
non_target_sorted = sorted(non_target, key=lambda n: -abs(n.get("influence") or 0))

print(f"\nTop 10 influential nodes:")
for node in non_target_sorted[:10]:
    nid = node.get("node_id")
    inf = node.get("influence")
    ftype = node.get("feature_type")
    print(f"  {nid}: influence={inf:.4f}, type={ftype}")

GRAPH STRUCTURE INSPECTION

Top-level keys: ['metadata', 'qParams', 'nodes', 'links']

Metadata: {'slug': 'step-19-it', 'scan': 'gemma-scope-2-270m-pt', 'transcoder_list': [], 'prompt_tokens': ['<bos>', 'A', ' rh', 'yming', ' couple', 't', ':', '\n', 'He', ' saw', ' a', ' carrot', ' and', ' had', ' to', ' grab', ' it', ',', '\n', 'He', ' saw', ' a', ' carrot', ' and', ' had', ' to', ' grab', ' it', ',', '\n', 'He', ' saw', ' a', ' carrot', ' and', ' had', ' to', ' grab'], 'prompt': '<bos>A rhyming couplet:\nHe saw a carrot and had to grab it,\nHe saw a carrot and had to grab it,\nHe saw a carrot and had to grab', 'node_threshold': 0.8, 'schema_version': 1}

# Nodes: 838
# Edges: 0

SAMPLE NODES

Node 0:
  node_id: 13_8735_37
  feature: 38276861
  layer: 13
  ctx_idx: 37
  feature_type: cross layer transcoder
  token_prob: 0.0
  is_target_logit: False
  run_idx: 0
  reverse_ctx_idx: 0
  jsNodeId: 13_8735-0
  clerp: 
  influence: 0.5741214156150818
  activation: 815104.0

Node 1:
  node_

In [45]:
import json
from pathlib import Path

GRAPH_DIR = "./graphs/gemma-3-270m"

with open(f"{GRAPH_DIR}/step-19-it.json") as f:
    data = json.load(f)

nodes = data.get("nodes", [])

print("=" * 70)
print("UNDERSTANDING ctx_idx (which token position features come from)")
print("=" * 70)

# The ctx_idx tells us WHICH TOKEN POSITION each feature is computed from
# ctx_idx=37 means "position 37 in the full sequence"
# ctx_idx=35 means "position 35"
# etc.

# Let's map: which ctx_idx values appear in high-influence nodes?
transcoder_nodes = [n for n in nodes 
                   if n.get("feature_type") == "cross layer transcoder"
                   and not n.get("is_target_logit")]

print(f"Transcoder nodes (the actual features): {len(transcoder_nodes)}")

# Group by ctx_idx to see which token positions matter
from collections import defaultdict
ctx_to_nodes = defaultdict(list)

for node in transcoder_nodes:
    ctx_idx = node.get("ctx_idx")
    ctx_to_nodes[ctx_idx].append(node)

print(f"\nToken positions that have influential features:")
for ctx_idx in sorted(ctx_to_nodes.keys()):
    nodes_at_pos = ctx_to_nodes[ctx_idx]
    avg_influence = sum(n.get("influence", 0) for n in nodes_at_pos) / len(nodes_at_pos)
    top_inf = max(n.get("influence", 0) for n in nodes_at_pos)
    print(f"  Position {ctx_idx:2d}: {len(nodes_at_pos):3d} features, "
          f"avg_inf={avg_influence:.4f}, max_inf={top_inf:.4f}")

print("\n" + "=" * 70)
print("NOW: MAP CONTEXT POSITIONS BACK TO GENERATION STEPS")
print("=" * 70)

# The base prompt is 19 tokens
# So: ctx_idx=19 = step 0 (first generated token)
#     ctx_idx=20 = step 1
#     ctx_idx=27 = step 8
#     ctx_idx=37 = step 18
#     ctx_idx=38 = step 19 (the rhyme we're predicting)

base_len = 19  # From your earlier output

print(f"\nBase prompt length: {base_len} tokens")
print(f"Generation steps: step 0 = position {base_len}, step 19 = position {base_len + 19}")

print("\nMapping influential positions to generation steps:")
for ctx_idx in sorted(ctx_to_nodes.keys()):
    step = ctx_idx - base_len
    nodes_at_pos = ctx_to_nodes[ctx_idx]
    avg_influence = sum(n.get("influence", 0) for n in nodes_at_pos) / len(nodes_at_pos)
    marker = ""
    if step == 9:
        marker = "  ← STEP 9 (comma)"
    elif step == 8:
        marker = "  ← STEP 8 (first 'it')"
    elif step == 19:
        marker = "  ← STEP 19 (second 'it', what we're predicting)"
    
    print(f"  Position {ctx_idx:2d} = Step {step:2d}  avg_inf={avg_influence:.4f}{marker}")

print("\n" + "=" * 70)
print("WHICH STEPS DRIVE THE RHYME?")
print("=" * 70)

# Find the steps with highest average influence
step_influence = {}
for ctx_idx, nodes_at_pos in ctx_to_nodes.items():
    step = ctx_idx - base_len
    avg_inf = sum(n.get("influence", 0) for n in nodes_at_pos) / len(nodes_at_pos)
    step_influence[step] = avg_inf

top_steps = sorted(step_influence.items(), key=lambda x: -x[1])

print("\nTop 10 steps by average feature influence on the rhyme:")
for step, avg_inf in top_steps[:10]:
    print(f"  Step {step:2d}: {avg_inf:.4f}")

print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)

# Is step 9 in the top?
if any(step == 9 for step, _ in top_steps[:5]):
    print("✓ Step 9 (comma) is in TOP 5 influential steps → planning signal!")
elif any(step == 9 for step, _ in top_steps[:10]):
    print("~ Step 9 (comma) is in TOP 10 but not top 5 → moderate planning")
else:
    print("✗ Step 9 (comma) is NOT in top 10 → step 9 features may not be causal")

UNDERSTANDING ctx_idx (which token position features come from)
Transcoder nodes (the actual features): 599

Token positions that have influential features:
  Position 37: 599 features, avg_inf=0.6064, max_inf=0.8002

NOW: MAP CONTEXT POSITIONS BACK TO GENERATION STEPS

Base prompt length: 19 tokens
Generation steps: step 0 = position 19, step 19 = position 38

Mapping influential positions to generation steps:
  Position 37 = Step 18  avg_inf=0.6064

WHICH STEPS DRIVE THE RHYME?

Top 10 steps by average feature influence on the rhyme:
  Step 18: 0.6064

CONCLUSION
✗ Step 9 (comma) is NOT in top 10 → step 9 features may not be causal


In [6]:
import torch
from itertools import product

prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

print("=" * 70)
print("BASELINE (no intervention)")
print("=" * 70)
baseline = model.feature_intervention_generate(
    prompt,
    [],
    do_sample=False,
)[0]
print(baseline)
baseline_end = baseline[-60:].strip()
print()

print("=" * 70)
print("TEMPORAL ABLATION: Suppress ALL features at each step")
print("=" * 70)
print()

critical_steps = []

for test_step in range(0, 20):
    # Create interventions: zero out ALL features at this step
    # All 18 layers × all 16384 features = too many
    # Instead: create a representative set
    
    intervention_tuples = []
    for layer in range(18):
        # Sample every Nth feature to avoid too many interventions
        for feat in range(0, 16384, 512):  # Every 512th feature
            intervention_tuples.append((layer, test_step, feat, 0.0))
    
    try:
        suppressed = model.feature_intervention_generate(
            prompt,
            intervention_tuples,
            do_sample=False,
        )[0]
        
        suppressed_end = suppressed[-60:].strip()
        has_effect = (suppressed_end != baseline_end)
        
        marker = "✓ EFFECT" if has_effect else "✗ no effect"
        print(f"Step {test_step:2d}: {marker}")
        
        if has_effect:
            critical_steps.append(test_step)
            # Show what changed
            print(f"  Baseline ends:   {repr(baseline_end[-40:])}")
            print(f"  Suppressed ends: {repr(suppressed_end[-40:])}")
        
    except Exception as e:
        print(f"Step {test_step:2d}: ERROR - {str(e)[:60]}")

print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)

if critical_steps:
    print(f"\n✓ Critical steps (suppression breaks output): {critical_steps}")
    if len(critical_steps) == 1:
        print(f"  → Planning/execution happens at step {critical_steps[0]}")
    elif max(critical_steps) - min(critical_steps) <= 2:
        print(f"  → Tight planning window: steps {min(critical_steps)}-{max(critical_steps)}")
    else:
        print(f"  → Distributed: planning spans steps {min(critical_steps)}-{max(critical_steps)}")
else:
    print("\n✗ No critical steps found")
    print("  Possible reasons:")
    print("  - Sampling strategy (every 512th feature) misses important features")
    print("  - Planning features are too distributed")
    print("  - Need to suppress ALL features at once (more aggressive)")

BASELINE (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]

A rhyming couplet:
He saw a carrot and had to grab it,
He saw a carrot and had to grab it,

TEMPORAL ABLATION: Suppress ALL features at each step



  0%|          | 0/10 [00:00<?, ?it/s]

Step  0: ✗ no effect


...

Step 17: ✓ EFFECT
  Baseline ends:   ' it,\nHe saw a carrot and had to grab it,'
  Suppressed ends: 'ad to grab it,\nHe had to\na carrot\nThe\nof'


  0%|          | 0/10 [00:00<?, ?it/s]

Step 18: ✓ EFFECT
  Baseline ends:   ' it,\nHe saw a carrot and had to grab it,'
  Suppressed ends: 'rab it,\n\nHe saw a carrot and had to grab'


  0%|          | 0/10 [00:00<?, ?it/s]

Step 19: ERROR - index 19 is out of bounds for dimension 0 with size 19

SUMMARY

✓ Critical steps (suppression breaks output): [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
  → Distributed: planning spans steps 1-18
